<a href="https://colab.research.google.com/github/Riccardo-27/DL-Quantum/blob/dev-riccardo/notebooks/04_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 – Modelli Transformer: Approccio "Start Small", Autoencoder (Lab 6), Architetture M3/M4 (Lab 7) e Sfide 1–2–3

Questo notebook implementa, addestra e valuta la suite di **modelli Transformer basati su Self-Attention e Cross-Attention** per la previsione multi-orizzonte ($T_{out} = 40$ passi temporali, $\Delta t = 0.02$, span $0.80$ unità fisiche) della dinamica quantistica a 10 qubit (55 osservabili continue in $[-1, 1]$).

In rigorosa conformità con le linee guida del docente e con i notebook di laboratorio (**Lab 6 – Autoencoders** e **Lab 7 – Attention and Transformers**), adottiamo un **processo graduale ed incrementale**:

1. **Fase Pilota "Start Small" e Riduzione della Dimensionalità (Lab 6 `Autoencoder.ipynb`)**:
   - *Inizia in piccolo*: partiamo da un **sottoinsieme limitato di dati** (20 traiettorie), **finestre temporali brevi** (`short`, $T_{in}=10$) e **modelli Transformer ridotti** ($d_{\text{model}}=32$, 1 layer, 2 teste) per sole **5 epoche** (`2 TF + 2 MM + 1 SS`), verificando in pochi secondi la convergenza del ciclo ibrido.
   - *Riduzione della dimensionalità tramite Autoencoder (Lab 6)*: addestriamo un `simple_autoencoder` (`Dense(16, activation='relu')` $\to$ `Dense(55, activation='tanh')`) per comprimere le 55 osservabili in una **rappresentazione latente compatta a $d_{\text{lat}} = 16$ dimensioni**, addestrando il Transformer direttamente nello spazio latente e confrontando il trade-off velocità/accuratezza rispetto allo spazio fisico completo a 55 variabili.
2. **Due Famiglie Architetturali del Lab 7 (`Transformer.ipynb`)**:
   - **M3 – Transformer Diretto (`TransformerDirect`, One-Shot)**: stack `Encoder` con `GlobalSelfAttention` bidirezionale sul passato $X \in (T_{in}, 55)$ seguito da una testa densa che emette in un colpo solo tutte le $T_{out} \times 55$ predizioni con attivazione $\tanh$. Immune per costruzione all'*exposure bias*.
   - **M4 – Transformer Seq2Seq Causale (`TransformerSeq2Seq`, Encoder-Decoder)**: architettura `Encoder` + `Decoder` completa come nel Lab 7 (`Transformer`), dotata di `CausalSelfAttention` (`use_causal_mask=True`) e `CrossAttention`.
3. **Scaling Graduale su 3 Configurazioni di Iperparametri (HP1, HP2, HP3) e Loop Ibrido (TF $\to$ MM $\to$ SS)**:
   - Valutazione sistematica su entrambe le finestre standard `short` ($T_{in}=10$) e `long` ($T_{in}=20$) variando `d_model`, `num_heads`, `d_ff`, `n_layers` e `dropout`.
4. **Risoluzione delle 3 Sfide Progettuali**:
   - **Sfida 1 (Finestra Minima Ottima)**: scaling dell'errore su $T_{in} \in \{5, 10, 20, 40\}$ con target rigorosamente allineati (`align_t_in=40`).
   - **Sfida 2 (Super-Risoluzione Temporale $\times 2$)**: previsione del futuro denso nativo ($\Delta t = 0.02$, 40 passi) da contesto passato sottocampionato ($\Delta t = 0.04$, 20 punti radi).
   - **Sfida 3 (Interpretabilità Fisica tramite Mappe di Attenzione)**: estrazione eager dei pesi di `CrossAttention` $(T_{out} \times T_{in})$ per ciascuna testa e verifica formale della maschera triangolare inferiore di `CausalSelfAttention` $(T_{out} \times T_{out})$.
5. **Validazione Statistica su 3 Seed ($t$ di Student) e Analisi Disaggregata**:
   - Repliche sui seed `[42, 43, 44]` (riutilizzando il `seed=42` già addestrato per non sprecare calcolo) con intervallo di confidenza al 95% ($t$ di Student) e salvataggio del checkpoint ottimale in `checkpoints/best_transformer_model.keras`.

## 0. Setup dell'Ambiente, Rilevamento Autore da Segreto Colab e Sincronizzazione Git

Rilevamento automatico dell'ambiente di esecuzione (Google Colab con Drive vs Locale).
In base al nome del segreto Colab configurato nell'icona 🔑 a sinistra:
- **`GIT_E_TOKEN`**: configura automaticamente l'autore **Ernesto** e sincronizza il branch **`dev-ernesto`**.
- **`GIT_R_TOKEN`**: configura automaticamente l'autore **Riccardo** e sincronizza il branch **`dev-riccardo`**.

Imposta inoltre il determinismo numerico multi-framework (`SEED = 42`) e verifica la disponibilità della GPU.

In [ ]:
# =============================================================================
# 0. Setup Ambiente, Rilevamento Autore da Segreto Colab e Sincronizzazione Git
# =============================================================================
import os
import sys
import time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

AUTHOR = None
CURRENT_BRANCH = None
GIT_TOKEN = None
AUTHOR_NAME = None
AUTHOR_EMAIL = None

if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")

    DRIVE_REPO = Path("/content/drive/MyDrive/DL-Quantum")
    LOCAL_REPO = Path("/content/DL-Quantum")

    if (DRIVE_REPO / "src" / "seeding.py").exists():
        REPO_PATH = DRIVE_REPO
    elif (LOCAL_REPO / "src" / "seeding.py").exists():
        REPO_PATH = LOCAL_REPO
    elif DRIVE_REPO.exists():
        !git -C /content/drive/MyDrive/DL-Quantum fetch origin
        REPO_PATH = DRIVE_REPO
    else:
        print("Clonazione automatica del repository su Google Drive...")
        !git clone https://github.com/Riccardo-27/DL-Quantum.git /content/drive/MyDrive/DL-Quantum
        REPO_PATH = DRIVE_REPO

    # 1. Rilevamento automatico dell'Autore dai Segreti di Colab (icona 🔑)
    # - Ernesto: imposta GIT_E_TOKEN nei propri Secret -> dev-ernesto
    # - Riccardo: imposta GIT_R_TOKEN nei propri Secret -> dev-riccardo
    try:
        GIT_TOKEN = userdata.get("GIT_E_TOKEN")
        if GIT_TOKEN:
            AUTHOR = "ernesto"
            CURRENT_BRANCH = "dev-ernesto"
            AUTHOR_NAME = "Ernesto Gligora"
            AUTHOR_EMAIL = "ernestogligora2003@gmail.com"
    except Exception:
        pass

    if not GIT_TOKEN:
        try:
            GIT_TOKEN = userdata.get("GIT_R_TOKEN")
            if GIT_TOKEN:
                AUTHOR = "riccardo"
                CURRENT_BRANCH = "dev-riccardo"
                AUTHOR_NAME = "Riccardo"
                AUTHOR_EMAIL = "lavoratoriccardo@gmail.com"
        except Exception:
            pass

    # Fallback su chiave generica GH_TOKEN se presente
    if not GIT_TOKEN:
        for k in ("GH_TOKEN", "GITHUB_TOKEN"):
            try:
                t = userdata.get(k)
                if t:
                    GIT_TOKEN = t
                    break
            except Exception:
                pass
else:
    # Ambiente Locale
    REPO_PATH = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

# Fallback se non rilevato dai segreti Colab (es. esecuzione locale da terminale)
if not CURRENT_BRANCH:
    b_check = !git -C "{REPO_PATH}" branch --show-current
    CURRENT_BRANCH = b_check[0].strip() if b_check and b_check[0].strip() else "dev-riccardo"

if not AUTHOR:
    if "riccardo" in CURRENT_BRANCH.lower():
        AUTHOR = "riccardo"
        AUTHOR_NAME = "Riccardo"
        AUTHOR_EMAIL = "lavoratoriccardo@gmail.com"
    else:
        AUTHOR = "ernesto"
        AUTHOR_NAME = "Ernesto Gligora"
        AUTHOR_EMAIL = "ernestogligora2003@gmail.com"

# Su Colab, allineamento automatico sul branch corrispondente all'autore rilevato
if IN_COLAB:
    print(f"Allineamento automatico sul branch '{CURRENT_BRANCH}' per {AUTHOR_NAME}...")
    !git -C "{REPO_PATH}" config user.name "{AUTHOR_NAME}"
    !git -C "{REPO_PATH}" config user.email "{AUTHOR_EMAIL}"
    !git -C "{REPO_PATH}" fetch origin
    !git -C "{REPO_PATH}" checkout -f {CURRENT_BRANCH}
    !git -C "{REPO_PATH}" pull --rebase --autostash origin {CURRENT_BRANCH}

# Configurazione variabili d'ambiente e path dei moduli src/
os.environ["DLQ_AUTHOR"] = AUTHOR
os.environ["DLQ_ROOT"] = str(REPO_PATH)

if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))

# Inizializzazione rigorosa dei seed per determinismo numerico
from src.seeding import set_seeds
SEED = 42
set_seeds(SEED)

try:
    import tensorflow as tf
    print("TensorFlow:", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))
except ImportError:
    print("TensorFlow non rilevato nell'ambiente locale")

token_status = "Rilevato ✔" if GIT_TOKEN else "Non impostato (crea segreto GIT_E_TOKEN o GIT_R_TOKEN per push automatico)"
print(f"\n✔ Ambiente inizializzato con successo!")
print(f"  • Autore attivo: {AUTHOR_NAME} ({token_status})")
print(f"  • Branch Git: {CURRENT_BRANCH}")
print(f"  • Root cartella: {REPO_PATH}")

In [ ]:
# =============================================================================
# Import Moduli di Progetto e Configurazione Percorsi
# =============================================================================
import importlib
import keras
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import src.models_transformer
importlib.reload(src.models_transformer)

from src.config import (
    DT, N_FEATURES, N_POINTS, N_QUBITS, N_TRAJ, get_paths, make_config
)
from src.data import (
    check_quantum_probabilities, feature_names, kfold_splits, load_split, load_trajectories
)
from src.models_transformer import (
    CUSTOM_OBJECTS, TransformerDirect, TransformerSeq2Seq,
    build_transformer_direct, build_transformer_from_config, positional_encoding
)
from src.training import train_hybrid, predict, is_seq2seq
from src.metrics import (
    evaluate, per_horizon_mse, per_horizon_group_mse,
    per_variable_mse, per_qubit_mse, corr_pair_mse_matrix
)
from src.stats import summarize, format_mean_ci
from src.results_log import append_result, load_results, log_path
from src.plots import save_figure, plot_window
from src.windowing import make_windows, make_sr_windows, decoder_inputs, check_no_leakage

CFG = make_config(model="transformer")
PATHS = get_paths()
NAMES = feature_names()

print("Configurazione centralizzata caricata con successo:")
print(f"  Orizzonte futuro (T_out)     : {CFG['t_out']} passi ({CFG['t_out']*DT:.2f} s)")
print(f"  Finestrature standard        : {CFG['windows']}")
print(f"  Fasi train loop ibrido       : TF={CFG['epochs_tf']}, MM={CFG['epochs_mm']}, SS={CFG['epochs_ss']} epoche")
print(f"  Attivazione output fisica    : {CFG['output_activation']} (dominio [-1, 1])")
print(f"  File di log autore           : {log_path()}")

## 1. Caricamento Dataset e Partizioni Congelate

Carichiamo il tensore 3D `(400, 1001, 55)` dalla cache binaria `data/cache/trajectories.npy` e gli indici di split congelati (280 Train, 60 Val, 60 Test) da `results/split_indices.npz`.

In [ ]:
# =============================================================================
# 1. Caricamento Dati e Split Congelato
# =============================================================================
t0 = time.time()
data = load_trajectories()
load_time = time.time() - t0

split = load_split()
print(f"Dataset caricato in {load_time:.2f} s | Shape tensore: {data.shape} (float32)")
print(f"Partizioni: Train={len(split['train'])}, Val={len(split['val'])}, Test={len(split['test'])}")
assert data.shape == (N_TRAJ, N_POINTS, N_FEATURES), "Shape dataset non conforme!"
assert len(split["train"]) + len(split["val"]) + len(split["test"]) == N_TRAJ, "Partizione incompleta!"
print("Integrita' dati e partizioni: verificata con successo ✔")

## 2. Preparazione Finestrature Standard (`short` e `long`) e Baseline Banali

Per non dover processare intere sequenze lunghe 1001 punti temporali, applichiamo la finestratura scorrevole (definita nel Notebook 02) con due orizzonti di contesto passato brevi:
- **`short`**: $T_{in} = 10$ ($0.20$ s) $\to T_{out} = 40$ ($0.80$ s), $\text{stride} = 10$.
- **`long`**: $T_{in} = 20$ ($0.40$ s) $\to T_{out} = 40$ ($0.80$ s), $\text{stride} = 10$.

Calcoliamo inoltre le **baseline banali non parametriche** (Persistenza dell'ultimo stato osservato $X[:, -1:, :]$ e Media globale del Training Set) come riferimento inferiore di accuratezza.

In [ ]:
# =============================================================================
# 2. Estrazione Finestre Standard (short: Tin=10, long: Tin=20) e Baseline Banali
# =============================================================================
windows_data = {}

for w_name, t_in in CFG["windows"].items():
    t0 = time.time()
    X_tr, Y_tr, meta_tr = make_windows(data, split["train"], t_in=t_in, t_out=CFG["t_out"], stride=CFG["stride"])
    X_va, Y_va, meta_va = make_windows(data, split["val"], t_in=t_in, t_out=CFG["t_out"], stride=CFG["stride"])
    X_te, Y_te, meta_te = make_windows(data, split["test"], t_in=t_in, t_out=CFG["t_out"], stride=CFG["stride"])

    # Verifica formale anti-leakage
    check_no_leakage({"train": meta_tr, "val": meta_va, "test": meta_te})

    windows_data[w_name] = {
        "t_in": t_in,
        "train": (X_tr, Y_tr),
        "val": (X_va, Y_va),
        "test": (X_te, Y_te),
    }
    el = time.time() - t0
    print(f"Finestra '{w_name:5s}' (Tin={t_in:2d}) estratta in {el:.2f} s:")
    print(f"  Train: X={X_tr.shape}, Y={Y_tr.shape} ({len(X_tr)} finestre)")
    print(f"  Val  : X={X_va.shape}, Y={Y_va.shape} ({len(X_va)} finestre)")
    print(f"  Test : X={X_te.shape}, Y={Y_te.shape} ({len(X_te)} finestre)")

# Baseline di riferimento (Persistenza e Media Training) su finestra 'short'
X_tr_s, Y_tr_s = windows_data["short"]["train"]
X_va_s, Y_va_s = windows_data["short"]["val"]

y_persist_va = np.repeat(X_va_s[:, -1:, :], CFG["t_out"], axis=1)
y_mean_va = np.broadcast_to(Y_tr_s.mean(axis=(0, 1), keepdims=True), Y_va_s.shape)
ev_persist = evaluate(Y_va_s, y_persist_va)
ev_mean = evaluate(Y_va_s, y_mean_va)

print("\n=== BASELINE DI RIFERIMENTO (Validation Set, Finestra 'short') ===")
print(f"  • Baseline Persistenza (ultimo step costante) : Val MSE = {ev_persist['mse']:.6f} | Val MAE = {ev_persist['mae']:.6f}")
print(f"  • Baseline Media Globale Training             : Val MSE = {ev_mean['mse']:.6f} | Val MAE = {ev_mean['mae']:.6f}")
print("Verifiche anti-leakage completate: zero contaminazione tra partizioni ✔")

## 3. Approccio Incrementale ("Start Small") e Riduzione della Dimensionalità con Autoencoder (Lab 6)

Seguendo alla lettera le indicazioni progettuali del docente:
> *"Inizia in PICCOLO! Parti da un sottoinsieme limitato di dati, sequenze e modelli di dimensioni ridotte! È un processo graduale. Non è necessario fare tutto subito! Sequenze troppo lunghe? Accorciale e utilizza finestre temporali più brevi! Troppe variabili? Applica una riduzione della dimensionalità (ad esempio, addestra un autoencoder e utilizza rappresentazioni latenti più compatte dei punti dati)."*

Prima di lanciare l'addestramento sull'intero dataset, eseguiamo due esperimenti pilota rapidissimi:
1. **Esperimento 3A — Pilot "Start Small" (Subset di 20 traiettorie, $T_{in}=10$, modello micro per 5 epoche)**:
   - Usiamo solo **20 traiettorie di training** ($1.920$ finestre) e **10 di validazione** ($960$ finestre) sulla finestra breve `short` ($T_{in}=10$).
   - Istanziamo un Transformer ridotto ($d_{\text{model}}=32$, $H=2$ teste, $d_{ff}=64$, $L=1$ layer) e lo addestriamo per sole **5 epoche** (`epochs_tf=2, epochs_mm=2, epochs_ss=1`) sia per **M3 (`TransformerDirect`)** che per **M4 (`TransformerSeq2Seq`)**, verificando che già in pochi secondi batta nettamente le baseline banali.
2. **Esperimento 3B — Riduzione della Dimensionalità con Autoencoder (Lab 6 `Autoencoder.ipynb`)**:
   - Come visto nel **Lab 6 (`Autoencoder.ipynb`)**, costruiamo un `simple_autoencoder` con `Input(shape=(55,))` $\to$ encoder `Dense(encoding_dim=16, activation='relu')` $\to$ decoder `Dense(55, activation='tanh')` che comprime le 55 osservabili fisiche in un **codice latente compatto a $16$ dimensioni** (compressione $\approx 3.4\times$).
   - Codifichiamo le finestre nello spazio latente $(T_{in}, 16)$, addestriamo un Transformer leggero per prevedere l'evoluzione latente futura $(T_{out}, 16)$ e decodifichiamo le predizioni nello spazio fisico originario $(T_{out}, 55)$, valutando il compromesso tra velocità/compattezza e precisione sulle correlazioni deboli.

In [ ]:
# =============================================================================
# 3A. "START SMALL": Sottoinsieme di 20 Traiettorie, Finestra Breve (Tin=10),
#     Modello Ridotto (d_model=32, L=1, H=2) e Poche Epoche (2 TF + 2 MM + 1 SS)
# =============================================================================
small_train_idx = split["train"][:20]  # Solo 20 traiettorie su 280 (~7% dei dati)
small_val_idx = split["val"][:10]      # Solo 10 traiettorie di validazione

X_tr_sm, Y_tr_sm, _ = make_windows(data, small_train_idx, t_in=10, t_out=40, stride=10)
X_va_sm, Y_va_sm, _ = make_windows(data, small_val_idx,   t_in=10, t_out=40, stride=10)

print(f"Subset 'Start Small' estratto: Train={X_tr_sm.shape} (20 traj), Val={X_va_sm.shape} (10 traj)")

small_histories = {}
for m_code, m_label in [("transformer_direct", "M3_Direct_Small"), ("transformer", "M4_Seq2Seq_Small")]:
    cfg_small = make_config(
        model=m_code,
        t_in=10,
        transformer_d_model=32,
        transformer_heads=2,
        transformer_d_ff=64,
        transformer_layers=1,
        dropout=0.0,
        epochs_tf=2,
        epochs_mm=2,
        epochs_ss=1,
        batch_size=64,
        run_name=m_label,
    )
    set_seeds(SEED)
    model_sm = build_transformer_from_config(cfg_small, t_in=10)
    t0 = time.time()
    hist_sm = train_hybrid(model_sm, X_tr_sm, Y_tr_sm, X_va_sm, Y_va_sm, cfg_small, verbose=0)
    dt_sm = time.time() - t0
    small_histories[m_label] = hist_sm

    ev_sm = evaluate(Y_va_sm, predict(model_sm, X_va_sm))
    print(
        f"  ✔ {m_label:18s} | Params: {model_sm.count_params():6,d} | 5 epoche in {dt_sm:4.1f} s | "
        f"Val MSE iniziale -> finale: {hist_sm[0]['val_mse']:.6f} -> {ev_sm['mse']:.6f} "
        f"(vs Persistenza {ev_persist['mse']:.6f})"
    )

In [ ]:
# =============================================================================
# 3B. RIDUZIONE DELLA DIMENSIONALITA' CON AUTOENCODER (Lab 6 - Autoencoder.ipynb)
#     Compressione 55 osservabili -> 16 variabili latenti + Transformer Latente
# =============================================================================
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model

encoding_dim = 16  # Spazio latente compatto: da 55 a 16 variabili (compressione 3.44x)
set_seeds(SEED)

# 1. Costruzione dell'Autoencoder esattamente come nel Lab 6 (Autoencoder.ipynb)
input_obs = Input(shape=(N_FEATURES,), name="input_observables")
code_layer = Dense(encoding_dim, activation="relu", name="latent_code")
decoder_layer = Dense(N_FEATURES, activation="tanh", name="decoded_observables")

code = code_layer(input_obs)
decoded = decoder_layer(code)

autoencoder = Model(input_obs, decoded, name="simple_autoencoder")
ae_encoder = Model(input_obs, code, name="ae_encoder")

latent_input = Input(shape=(encoding_dim,), name="latent_input")
ae_decoder = Model(latent_input, decoder_layer(latent_input), name="ae_decoder")

autoencoder.compile(optimizer="adam", loss="mse")

# Addestramento rapido dell'Autoencoder sui punti temporali del subset Start Small
x_ae_train = data[small_train_idx].reshape(-1, N_FEATURES)
x_ae_val = data[small_val_idx].reshape(-1, N_FEATURES)

t0_ae = time.time()
ae_hist = autoencoder.fit(
    x_ae_train, x_ae_train,
    epochs=10,
    batch_size=256,
    validation_data=(x_ae_val, x_ae_val),
    verbose=0,
)
t_ae = time.time() - t0_ae
recon_mse = float(ae_hist.history["val_loss"][-1])

# 2. Proiezione delle finestre temporali nello spazio latente compatto (n_win, t, 16)
Z_tr_sm = ae_encoder.predict(X_tr_sm.reshape(-1, N_FEATURES), batch_size=512, verbose=0).reshape(len(X_tr_sm), 10, encoding_dim)
Z_target_tr_sm = ae_encoder.predict(Y_tr_sm.reshape(-1, N_FEATURES), batch_size=512, verbose=0).reshape(len(Y_tr_sm), 40, encoding_dim)
Z_va_sm = ae_encoder.predict(X_va_sm.reshape(-1, N_FEATURES), batch_size=512, verbose=0).reshape(len(X_va_sm), 10, encoding_dim)
Z_target_va_sm = ae_encoder.predict(Y_va_sm.reshape(-1, N_FEATURES), batch_size=512, verbose=0).reshape(len(Y_va_sm), 40, encoding_dim)

# 3. Addestramento del Transformer direttamente sulle rappresentazioni latenti (16 dim)
cfg_lat = make_config(
    model="transformer_direct",
    t_in=10,
    transformer_d_model=32,
    transformer_heads=2,
    transformer_d_ff=64,
    transformer_layers=1,
    dropout=0.0,
    output_activation="linear",  # Lo spazio latente ReLU non e' limitato a [-1, 1]
    epochs_tf=2,
    epochs_mm=2,
    epochs_ss=1,
    batch_size=64,
)
set_seeds(SEED)
model_lat = build_transformer_direct(
    t_out=40,
    t_in=10,
    n_features=encoding_dim,
    d_model=32,
    num_heads=2,
    d_ff=64,
    n_layers=1,
    dropout=0.0,
    output_activation="linear",
    name="M3_Latent_16d",
)
t0_lat = time.time()
hist_lat = train_hybrid(model_lat, Z_tr_sm, Z_target_tr_sm, Z_va_sm, Z_target_va_sm, cfg_lat, verbose=0)
t_lat = time.time() - t0_lat

# 4. Decodifica delle predizioni latenti nello spazio fisico a 55 osservabili
Z_pred_va = predict(model_lat, Z_va_sm)
Y_pred_from_lat = ae_decoder.predict(Z_pred_va.reshape(-1, encoding_dim), batch_size=512, verbose=0).reshape(Y_va_sm.shape)
ev_lat_phys = evaluate(Y_va_sm, Y_pred_from_lat)
params_full_55 = build_transformer_direct(t_out=40, t_in=10, n_features=N_FEATURES, d_model=32, num_heads=2, d_ff=64, n_layers=1).count_params()

print(f"Autoencoder (55 -> {encoding_dim} -> 55) addestrato in {t_ae:.1f} s | MSE ricostruzione pura: {recon_mse:.6f}")
print(
    f"Transformer Latente ({encoding_dim} dim) | Params: {model_lat.count_params():,d} (vs {params_full_55:,d} a 55 dim) | "
    f"Tempo: {t_lat:.1f} s | Val MSE fisico ricostruito: {ev_lat_phys['mse']:.6f}"
)

# Grafico riassuntivo "Start Small" + Autoencoder (Lab 6)
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.0))
for k_sm, h_sm in small_histories.items():
    axes[0].plot([r["epoch"] for r in h_sm], [r["val_mse"] for r in h_sm], "o-", lw=2, label=k_sm)
axes[0].axhline(ev_lat_phys["mse"], color="darkgreen", ls="--", lw=1.8, label=f"AE Latente 16-dim + Transformer ({ev_lat_phys['mse']:.5f})")
axes[0].set_title("Fase 'Start Small' (20 Traj, 5 Epoche): Discesa Val MSE")
axes[0].set_xlabel("Epoca (2 TF -> 2 MM -> 1 SS)")
axes[0].set_ylabel("Validation MSE (55 osservabili)")
axes[0].legend(fontsize=8.5)
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, len(ae_hist.history["loss"]) + 1), ae_hist.history["loss"], "bo-", label="Training loss AE")
axes[1].plot(range(1, len(ae_hist.history["val_loss"]) + 1), ae_hist.history["val_loss"], "rs--", label="Validation loss AE")
axes[1].set_title(f"Autoencoder Lab 6 (55 -> {encoding_dim} -> 55): Loss di Ricostruzione")
axes[1].set_xlabel("Epoca")
axes[1].set_ylabel("MSE di Ricostruzione")
axes[1].legend(fontsize=8.5)
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
print("Grafico esportato:", save_figure(fig, "start_small_and_autoencoder", subdir="04_transformer"))
plt.show()

## 4. Ispezione delle Architetture del Lab 7: M3 (`TransformerDirect`) vs M4 (`TransformerSeq2Seq`)

Verificata nella fase *Start Small* la bontà del ciclo di addestramento e osservato che lo spazio fisico diretto a 55 osservabili evita il pavimento dell'errore di ricostruzione dell'Autoencoder sulle correlazioni deboli, passiamo all'ispezione delle architetture complete del **Lab 7 (`Attention.ipynb` e `Transformer.ipynb`)**:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}} + M\right)V$$

dove l'informazione d'ordine temporale è fornita dalla **codifica posizionale sinusoidale** (`positional_encoding`) sommata alla proiezione lineare continua delle 55 osservabili (`PositionalEmbedding`).

Confrontiamo le due famiglie architetturali definite in `src/models_transformer.py`:
- **M3 – `TransformerDirect` (Encoder-Only One-Shot)**: stack `Encoder` (`GlobalSelfAttention` + `FeedForward`) che contestualizza l'intera finestra passata; l'ultimo token contestualizzato viene proiettato in un colpo solo sull'orizzonte $(T_{out}, 55)$ con attivazione $\tanh$.
- **M4 – `TransformerSeq2Seq` (Encoder-Decoder Causale Autoregressivo)**: riproduce la classe `Transformer` del Lab 7 (`Encoder` + `Decoder` con `CausalSelfAttention(use_causal_mask=True)` e `CrossAttention`). In training sfrutta la maschera causale per il *Teacher Forcing* parallelo; in inferenza genera un passo alla volta in *Free-Running* ad anello chiuso.

In [ ]:
# =============================================================================
# 4. Istanziazione e Ispezione di M3 (Direct) ed M4 (Seq2Seq Causale)
# =============================================================================
demo_tin = 10
demo_tout = 40

set_seeds(SEED)
m3_demo = build_transformer_direct(
    t_out=demo_tout, t_in=demo_tin, d_model=64, num_heads=4, d_ff=128, n_layers=2, dropout=0.1, name="M3_TransformerDirect"
)
print("=== ARCHITETTURA M3 (Transformer Diretto - Encoder-Only One-Shot) ===")
m3_demo.summary()

m4_demo = TransformerSeq2Seq(
    t_out=demo_tout, d_model=64, num_heads=4, d_ff=128, n_layers=2, dropout=0.1, name="M4_TransformerSeq2Seq"
)
_ = m4_demo((tf.zeros((1, demo_tin, N_FEATURES)), tf.zeros((1, demo_tout, N_FEATURES))))
print("\n=== ARCHITETTURA M4 (Transformer Seq2Seq Causale - Encoder-Decoder) ===")
m4_demo.summary()

# Verifica coerenza causale tra Free-Running e Teacher Forcing auto-alimentato (assenza di leakage futuro)
x_chk = X_va_s[:4]
y_fr_chk = m4_demo.predict_free_running(x_chk, batch_size=4)
y_tf_self = m4_demo((tf.constant(x_chk), tf.constant(decoder_inputs(x_chk, y_fr_chk))), training=False).numpy()
max_diff = float(np.max(np.abs(y_fr_chk - y_tf_self)))

print(f"\nParametri M3: {m3_demo.count_params():,} | Parametri M4: {m4_demo.count_params():,}")
print(f"Verifica invarianza causale M4 | max |FreeRunning - SelfFed_TF| = {max_diff:.2e} (< 1e-5) ✔")

## 5. Scaling sul Dataset Completo: 3 Configurazioni di Iperparametri (HP1, HP2, HP3)

Scaliamo ora gradualmente sul dataset completo valutando **3 configurazioni di iperparametri** su entrambe le finestre temporali (`short` e `long`) per ciascuna delle due architetture (**M3** e **M4**, 12 modelli), partendo dalla configurazione più compatta (**HP3**) fino a quella a maggiore capacità (**HP2**):

| Configurazione | `d_model` | `num_heads` | `d_ff` | `n_layers` | `dropout` | Motivazione Architetturale |
| :--- | :---: | :---: | :---: | :---: | :---: | :--- |
| **HP1 (Baseline)** | **64** | **4** | **128** | **2** | **0.1** | Architettura bilanciata a 4 teste d'attenzione e 2 blocchi impilati con regolarizzazione leggera |
| **HP2 (Alta Capacità)** | **128** | **4** | **256** | **2** | **0.1** | Raddoppio dello spazio di embedding (`d_model=128`) e del Feed-Forward (`d_ff=256`) per correlazioni complesse |
| **HP3 (Compatto)** | **64** | **2** | **128** | **1** | **0.0** | Singolo blocco Transformer a 2 teste senza dropout per verificare la parsimonia parametrica |

Tutti i modelli vengono addestrati con il **train loop ibrido unificato** (`train_hybrid`: 10 epoche TF + 10 epoche MM con `mask_ratio=0.15` + 5 epoche SS con rampa lineare $p: 0.0 \to 0.5$).

In [ ]:
# =============================================================================
# 5. Definizione Griglia Esperimenti HP1, HP2, HP3 x (short, long)
# =============================================================================
hp_configs = {
    "HP1": {"transformer_d_model": 64,  "transformer_heads": 4, "transformer_d_ff": 128, "transformer_layers": 2, "dropout": 0.1},
    "HP2": {"transformer_d_model": 128, "transformer_heads": 4, "transformer_d_ff": 256, "transformer_layers": 2, "dropout": 0.1},
    "HP3": {"transformer_d_model": 64,  "transformer_heads": 2, "transformer_d_ff": 128, "transformer_layers": 1, "dropout": 0.0},
}

models_to_test = [
    ("transformer_direct", "M3_Direct"),
    ("transformer",        "M4_Seq2Seq"),
]

print(f"Pianificati {len(models_to_test) * len(hp_configs) * len(CFG['windows'])} esperimenti Transformer:")
for m_type, m_name in models_to_test:
    for hp_name, hp_cfg in hp_configs.items():
        for w_name in CFG["windows"]:
            print(
                f"  - {m_name:10s} | {hp_name} (d={hp_cfg['transformer_d_model']}, "
                f"H={hp_cfg['transformer_heads']}, L={hp_cfg['transformer_layers']}, "
                f"ff={hp_cfg['transformer_d_ff']}, drop={hp_cfg['dropout']}) | Finestra '{w_name}'"
            )

In [ ]:
# =============================================================================
# Esecuzione Addestramento Ibrido e Valutazione HP su Finestre Standard
# =============================================================================
experiment_records = []
histories = {}
trained_models = {}

total_runs = len(models_to_test) * len(hp_configs) * len(CFG["windows"])
run_idx = 0

for w_name, w_info in windows_data.items():
    t_in = w_info["t_in"]
    X_tr, Y_tr = w_info["train"]
    X_va, Y_va = w_info["val"]

    for m_type, m_label in models_to_test:
        for hp_name, hp_kwargs in hp_configs.items():
            run_idx += 1
            exp_id = f"{m_label}_{hp_name}_{w_name}"
            print(f"\n[{run_idx}/{total_runs}] Addestramento: {exp_id} (Tin={t_in}, Tout=40)...")

            run_cfg = make_config(
                model=m_type,
                t_in=t_in,
                hp_name=hp_name,
                window_name=w_name,
                run_name=exp_id,
                **hp_kwargs
            )

            # Determinismo
            set_seeds(run_cfg["seed"])

            # Costruzione modello Transformer
            model = build_transformer_from_config(run_cfg, t_in=t_in)

            # Addestramento ibrido (TF -> MM -> SS)
            t_train_start = time.time()
            hist = train_hybrid(
                model=model,
                X_tr=X_tr, Y_tr=Y_tr,
                X_va=X_va, Y_va=Y_va,
                cfg=run_cfg,
                restore_best=True,
                verbose=0
            )
            train_duration = time.time() - t_train_start
            histories[exp_id] = hist
            trained_models[exp_id] = model

            # Predizione su validation (Free-Running se seq2seq, one-shot se diretta)
            Y_pred_val = predict(model, X_va)
            val_metrics = evaluate(Y_va, Y_pred_val, prefix="val_")

            # Record per log e tabella
            rec = {
                "model_type": m_type,
                "model_name": m_label,
                "hp_name": hp_name,
                "window": w_name,
                "t_in": t_in,
                "d_model": hp_kwargs["transformer_d_model"],
                "heads": hp_kwargs["transformer_heads"],
                "d_ff": hp_kwargs["transformer_d_ff"],
                "layers": hp_kwargs["transformer_layers"],
                "dropout": hp_kwargs["dropout"],
                "params": int(model.count_params()),
                "train_time_s": round(train_duration, 1),
                "best_val_loss": round(min(h["val_mse"] for h in hist), 6),
                **val_metrics
            }
            experiment_records.append(rec)
            append_result(rec)

            print(f"  ✔ Completato in {train_duration:.1f} s | Params={rec['params']:,} | Val MSE={val_metrics['val_mse']:.6f} "
                  f"(Mag={val_metrics['val_mse_mag']:.6f}, Corr={val_metrics['val_mse_corr']:.6f})")

# Visualizzazione tabella riassuntiva ordinata per Val MSE
results_df = pd.DataFrame(experiment_records)
cols_show = [
    "model_name", "hp_name", "window", "d_model", "heads", "layers",
    "dropout", "params", "train_time_s", "val_mse", "val_mae", "val_mse_mag", "val_mse_corr"
]
display(results_df[cols_show].sort_values("val_mse"))

## 6. Visualizzazione delle Curve di Apprendimento del Train Loop Ibrido

Analizziamo l'evoluzione dell'MSE di validazione attraverso le tre fasi del train loop ibrido (**TF $\to$ MM $\to$ SS**) per entrambe le famiglie **M3 (`TransformerDirect`)** e **M4 (`TransformerSeq2Seq`)**, evidenziando le transizioni tra Teacher Forcing, Masked Modeling e Scheduled Sampling.

In [ ]:
# =============================================================================
# 6. Grafico delle Curve di Apprendimento nelle 3 Fasi Ibride
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2), sharey=True)

for i, (m_type, m_label) in enumerate(models_to_test):
    ax = axes[i]
    for hp_name in hp_configs:
        exp_id = f"{m_label}_{hp_name}_short"
        if exp_id in histories:
            h = histories[exp_id]
            epochs = [r["epoch"] for r in h]
            val_mses = [r["val_mse"] for r in h]
            ax.plot(epochs, val_mses, marker="o", ms=3.5, lw=1.8, label=f"{hp_name}")

    # Linee verticali divisorie tra le fasi
    ax.axvline(10.5, color="gray", ls="--", lw=0.9, alpha=0.7)
    ax.axvline(20.5, color="gray", ls="--", lw=0.9, alpha=0.7)
    ax.text(5.5, ax.get_ylim()[1] * 0.9, "Fase 1: TF", ha="center", fontsize=8.5, color="navy")
    ax.text(15.5, ax.get_ylim()[1] * 0.9, "Fase 2: MM", ha="center", fontsize=8.5, color="darkgreen")
    ax.text(23.0, ax.get_ylim()[1] * 0.9, "Fase 3: SS", ha="center", fontsize=8.5, color="darkred")

    ax.set_title(f"{m_label} – Finestra 'short' (Tin=10)")
    ax.set_xlabel("Epoca")
    ax.set_ylabel("Validation MSE (scala log)")
    ax.set_yscale("log")
    ax.legend(fontsize=8.5)
    ax.grid(True, alpha=0.3)

fig.tight_layout()
print("Grafico esportato:", save_figure(fig, "loss_curves_hybrid", subdir="04_transformer"))
plt.show()

## 7. Confronto sull'Orizzonte Futuro: Teacher Forcing vs Free-Running (M4) vs One-Shot (M3)

Per il miglior modello autoregressivo **M4 (`TransformerSeq2Seq`)**, confrontiamo la propagazione dell'errore quadratico medio lungo i 40 passi futuri ($\tau \in \{1, \dots, 40\}$) in modalità **Teacher Forcing** (oracolo che riceve a ogni passo il target reale precedente), in **Free-Running** (inferenza reale a circuito chiuso) e rispetto alla proiezione diretta **M3 (`TransformerDirect`)**.

In [ ]:
# =============================================================================
# 7. Analisi Propagazione dell'Errore: Teacher Forcing vs Free-Running (M4) vs M3
# =============================================================================
df_short = results_df[results_df["window"] == "short"].sort_values("val_mse")
best_m3_row = df_short[df_short["model_type"] == "transformer_direct"].iloc[0]
best_m4_row = df_short[df_short["model_type"] == "transformer"].iloc[0]

best_m3_hp = best_m3_row["hp_name"]
best_m4_hp = best_m4_row["hp_name"]
m3_eval = trained_models[f"M3_Direct_{best_m3_hp}_short"]
m4_eval = trained_models[f"M4_Seq2Seq_{best_m4_hp}_short"]

X_va_short, Y_va_short = windows_data["short"]["val"]

# 1. Predizione M4 in Teacher Forcing (con target veri shiftati)
dec_in_val = decoder_inputs(X_va_short, Y_va_short)
Y_pred_m4_tf = np.concatenate([
    m4_eval(
        (tf.convert_to_tensor(X_va_short[i : i + 512], dtype=tf.float32),
         tf.convert_to_tensor(dec_in_val[i : i + 512], dtype=tf.float32)),
        training=False,
    ).numpy()
    for i in range(0, len(X_va_short), 512)
], axis=0)

# 2. Predizione M4 in Free-Running (autoregressivo a circuito chiuso)
Y_pred_m4_fr = m4_eval.predict_free_running(X_va_short, batch_size=512)

# 3. Predizione M3 Diretta One-Shot
Y_pred_m3_dir = predict(m3_eval, X_va_short, batch_size=512)

mse_tf_horizon = per_horizon_mse(Y_va_short, Y_pred_m4_tf)
mse_fr_horizon = per_horizon_mse(Y_va_short, Y_pred_m4_fr)
mse_m3_horizon = per_horizon_mse(Y_va_short, Y_pred_m3_dir)

fig, ax = plt.subplots(figsize=(9.5, 4.2))
time_axis = np.arange(1, CFG["t_out"] + 1) * DT

ax.plot(time_axis, mse_tf_horizon, label=f"M4 Seq2Seq ({best_m4_hp}) – Teacher Forcing (MSE={mse_tf_horizon.mean():.6f})", color="C0", lw=2)
ax.plot(time_axis, mse_fr_horizon, label=f"M4 Seq2Seq ({best_m4_hp}) – Free-Running (MSE={mse_fr_horizon.mean():.6f})", color="C3", lw=2, ls="--")
ax.plot(time_axis, mse_m3_horizon, label=f"M3 Direct ({best_m3_hp}) – One-Shot (MSE={mse_m3_horizon.mean():.6f})", color="C2", lw=2, ls="-.")

ratio = mse_fr_horizon.mean() / max(mse_tf_horizon.mean(), 1e-12)
ax.set_title(f"Propagazione dell'Errore lungo l'Orizzonte Futuro (Rapporto FR/TF M4: {ratio:.2f}x)")
ax.set_xlabel("Orizzonte di predizione futuro [tempo fisico s]")
ax.set_ylabel("MSE per timestep")
ax.legend(fontsize=8.5)
ax.grid(True, alpha=0.3)

fig.tight_layout()
print("Grafico esportato:", save_figure(fig, "free_running_error_accumulation", subdir="04_transformer"))
plt.show()

## 8. Risoluzione della Sfida 1: Ricerca della Finestra Minima Ottima ($T_{in} \in \{5, 10, 20, 40\}$)

Valutiamo lo scaling dell'errore predittivo del Transformer vincente al variare della lunghezza del contesto passato osservato $T_{in} \in \{5, 10, 20, 40\}$, mantenendo i target futuri rigorosamente allineati all'istante fisico $t = 0.80$ s tramite `align_t_in=40`.

In [ ]:
# =============================================================================
# 8. Esecuzione Griglia Tin in {5, 10, 20, 40} con Target Rigorosamente Allineati
# =============================================================================
tin_results = []
tin_grid = CFG["t_in_grid"]  # [5, 10, 20, 40]

# Usiamo la configurazione Transformer complessivamente vincente
best_overall_row = results_df.sort_values("val_mse").iloc[0]
best_model_type = best_overall_row["model_type"]
best_model_label = best_overall_row["model_name"]
best_hp = best_overall_row["hp_name"]
print(f"Risoluzione Sfida 1 con architettura vincente: {best_model_label} ({best_hp})...")

for tin in tin_grid:
    t0 = time.time()
    X_tr_aln, Y_tr_aln, _ = make_windows(data, split["train"], t_in=tin, t_out=40, stride=10, align_t_in=40)
    X_va_aln, Y_va_aln, _ = make_windows(data, split["val"],   t_in=tin, t_out=40, stride=10, align_t_in=40)
    X_te_aln, Y_te_aln, _ = make_windows(data, split["test"],  t_in=tin, t_out=40, stride=10, align_t_in=40)

    cfg_tin = make_config(
        model=best_model_type,
        t_in=tin,
        hp_name=f"Sfida1_Transformer_Tin_{tin}",
        align_t_in=40,
        **hp_configs[best_hp]
    )

    set_seeds(42)
    model_tin = build_transformer_from_config(cfg_tin, t_in=tin)

    _ = train_hybrid(model_tin, X_tr_aln, Y_tr_aln, X_va_aln, Y_va_aln, cfg_tin, verbose=0)

    val_res = evaluate(Y_va_aln, predict(model_tin, X_va_aln), prefix="val_")
    test_res = evaluate(Y_te_aln, predict(model_tin, X_te_aln), prefix="test_")

    row = {
        "model_type": best_model_type,
        "model_name": best_model_label,
        "hp_name": f"Sfida1_Tin_{tin}",
        "window": f"tin_{tin}_aligned",
        "T_in": tin,
        "tempo_fisico_in": tin * DT,
        "n_finestre_train": len(X_tr_aln),
        "val_mse": val_res["val_mse"],
        "val_mae": val_res["val_mae"],
        "val_mse_mag": val_res["val_mse_mag"],
        "val_mse_corr": val_res["val_mse_corr"],
        "test_mse": test_res["test_mse"],
        "test_mae": test_res["test_mae"],
        "elapsed_s": round(time.time() - t0, 1),
    }
    tin_results.append(row)
    append_result(row)
    print(f"  Tin={tin:2d} ({tin*DT:.2f} s) | Val MSE={val_res['val_mse']:.6f} | Test MSE={test_res['test_mse']:.6f} ({row['elapsed_s']} s)")

tin_df = pd.DataFrame(tin_results)
display(tin_df[["T_in", "tempo_fisico_in", "val_mse", "test_mse", "val_mse_mag", "val_mse_corr", "elapsed_s"]])

In [ ]:
# =============================================================================
# Grafico Sfida 1: Curva di Scaling dell'Errore in Funzione di Tin
# =============================================================================
fig, ax1 = plt.subplots(figsize=(8, 4.2))

ax1.plot(tin_df["T_in"], tin_df["val_mse"], marker="o", color="navy", lw=2, label="Validation MSE")
ax1.plot(tin_df["T_in"], tin_df["test_mse"], marker="s", color="crimson", lw=2, ls="--", label="Test MSE")
ax1.set_xlabel(r"Contesto di osservazione passata $T_{in}$ (passi temporali)")
ax1.set_ylabel("MSE (target allineati)")
ax1.set_xticks(tin_df["T_in"])
ax1.grid(True, alpha=0.3)

ax2 = ax1.twiny()
ax2.set_xlim(ax1.get_xlim())
ax2.set_xticks(tin_df["T_in"])
ax2.set_xticklabels([f"{t:.2f} s" for t in tin_df["tempo_fisico_in"]])
ax2.set_xlabel("Tempo fisico di osservazione passata")

ax1.legend(loc="upper right")
plt.title(r"Sfida 1 (Transformer) – Errore in Funzione della Lunghezza del Contesto ($T_{in}$)", pad=15)
fig.tight_layout()
print("Grafico esportato:", save_figure(fig, "challenge1_tin_scaling", subdir="04_transformer"))
plt.show()

## 9. Risoluzione della Sfida 2: Super-Risoluzione Temporale (Subsampling $\times 2$, $\Delta t = 0.04 \to 0.02$)

Nella **Sfida 2**, il modello riceve in input $T_{in} = 20$ punti passati **sottocampionati di un fattore $\times 2$** (passo $\Delta t_{sub} = 0.04$ s, che coprono uno span fisico continuo di $39$ punti densi nativi) e deve prevedere i successivi $T_{out} = 40$ passi futuri alla **piena risoluzione nativa ($\Delta t = 0.02$ s)**.

Addestriamo l'architettura Transformer vincente in regime di Super-Risoluzione e la confrontiamo direttamente con il riferimento addestrato sul corrispondente contesto denso nativo ($39$ punti a $\Delta t = 0.02$ s).

In [ ]:
# =============================================================================
# 9. Sfida 2: Super-Risoluzione Temporale (Subsampling x2, dt=0.04 -> dt=0.02)
# =============================================================================
sr_t_in = CFG["sr_t_in"]     # 20 punti radi
sr_factor = CFG["sr_factor"] # fattore di subsampling x2

X_sr_tr, Y_sr_tr, meta_sr_tr, X_dense_tr = make_sr_windows(
    data, split["train"], t_in=sr_t_in, t_out=40, stride=10, factor=sr_factor, return_dense_input=True
)
X_sr_va, Y_sr_va, meta_sr_va, X_dense_va = make_sr_windows(
    data, split["val"],   t_in=sr_t_in, t_out=40, stride=10, factor=sr_factor, return_dense_input=True
)
X_sr_te, Y_sr_te, meta_sr_te, X_dense_te = make_sr_windows(
    data, split["test"],  t_in=sr_t_in, t_out=40, stride=10, factor=sr_factor, return_dense_input=True
)
check_no_leakage({"train": meta_sr_tr, "val": meta_sr_va, "test": meta_sr_te})

print(f"Finestre Super-Risoluzione (x{sr_factor}) estratte per {best_model_label} ({best_hp}):")
print(f"  Input rado X_sr (dt=0.04)     : Train={X_sr_tr.shape}, Val={X_sr_va.shape}, Test={X_sr_te.shape}")
print(f"  Contesto denso X_dense (0.02) : Train={X_dense_tr.shape}, Val={X_dense_va.shape}, Test={X_dense_te.shape}")
print(f"  Target denso Y_sr (dt=0.02)   : Train={Y_sr_tr.shape}, Val={Y_sr_va.shape}, Test={Y_sr_te.shape}")

sr_records = []

# A. Addestramento su input rado sottocampionato (20 punti a dt=0.04)
t0 = time.time()
cfg_sr = make_config(
    model=best_model_type,
    t_in=sr_t_in,
    hp_name=f"Sfida2_{best_model_label}_SR_x2",
    window_name="super_res_x2",
    **hp_configs[best_hp]
)
set_seeds(42)
model_sr = build_transformer_from_config(cfg_sr, t_in=sr_t_in)
_ = train_hybrid(model_sr, X_sr_tr, Y_sr_tr, X_sr_va, Y_sr_va, cfg_sr, verbose=0)
el_sr = round(time.time() - t0, 1)

pred_va_sr_best = predict(model_sr, X_sr_va)
pred_te_sr_best = predict(model_sr, X_sr_te)
ev_va_sr = evaluate(Y_sr_va, pred_va_sr_best, prefix="val_")
ev_te_sr = evaluate(Y_sr_te, pred_te_sr_best, prefix="test_")

rec_sr = {
    "model_type": best_model_type,
    "model_name": f"{best_model_label}_SR_x2",
    "hp_name": best_hp,
    "window": "super_res_x2 (dt=0.04->0.02)",
    "t_in": sr_t_in,
    "train_time_s": el_sr,
    **ev_va_sr,
    "test_mse": ev_te_sr["test_mse"],
    "test_mae": ev_te_sr["test_mae"],
}
sr_records.append(rec_sr)
append_result(rec_sr)
print(f"  ✔ {best_model_label} Super-Res (20 pt, dt=0.04) | Val MSE={ev_va_sr['val_mse']:.6f} | Test MSE={ev_te_sr['test_mse']:.6f} ({el_sr} s)")

# B. Riferimento a risoluzione nativa densa sullo stesso span fisico (39 punti a dt=0.02)
t0 = time.time()
dense_tin = X_dense_tr.shape[1]
cfg_nat = make_config(
    model=best_model_type,
    t_in=dense_tin,
    hp_name=f"Sfida2_{best_model_label}_Native",
    window_name="native_dense39",
    **hp_configs[best_hp]
)
set_seeds(42)
model_nat = build_transformer_from_config(cfg_nat, t_in=dense_tin)
_ = train_hybrid(model_nat, X_dense_tr, Y_sr_tr, X_dense_va, Y_sr_va, cfg_nat, verbose=0)
el_nat = round(time.time() - t0, 1)

pred_va_nat_best = predict(model_nat, X_dense_va)
pred_te_nat_best = predict(model_nat, X_dense_te)
ev_va_nat = evaluate(Y_sr_va, pred_va_nat_best, prefix="val_")
ev_te_nat = evaluate(Y_sr_te, pred_te_nat_best, prefix="test_")

rec_nat = {
    "model_type": best_model_type,
    "model_name": f"{best_model_label}_Native_dt002",
    "hp_name": best_hp,
    "window": "native_dense (dt=0.02->0.02)",
    "t_in": dense_tin,
    "train_time_s": el_nat,
    **ev_va_nat,
    "test_mse": ev_te_nat["test_mse"],
    "test_mae": ev_te_nat["test_mae"],
}
sr_records.append(rec_nat)
append_result(rec_nat)
print(f"  ✔ {best_model_label} Nativo    (39 pt, dt=0.02) | Val MSE={ev_va_nat['val_mse']:.6f} | Test MSE={ev_te_nat['test_mse']:.6f} ({el_nat} s)")

sr_df = pd.DataFrame(sr_records)
display(sr_df[["model_name", "window", "t_in", "val_mse", "val_mae", "val_mse_mag", "val_mse_corr", "test_mse", "test_mae", "train_time_s"]])

In [ ]:
# =============================================================================
# Grafico Sfida 2: Ricostruzione Temporale Super-Risoluzione e Confronto Orizzonte
# =============================================================================
sample_idx = 0
feat_idx = 0  # m_1 = <sigma_z^1>
t_dense_past = np.arange(-X_dense_te.shape[1] + 1, 1) * DT
t_sparse_past = t_dense_past[::sr_factor]
t_future = np.arange(1, CFG["t_out"] + 1) * DT

fig, axes = plt.subplots(1, 2, figsize=(14, 4.4))

# Pannello A: Confronto segnale su singola finestra di test
axes[0].plot(t_dense_past, X_dense_te[sample_idx, :, feat_idx], "--", color="gray", alpha=0.65, label=r"Passato denso nascosto ($\Delta t=0.02$)")
axes[0].scatter(t_sparse_past, X_sr_te[sample_idx, :, feat_idx], color="navy", s=35, zorder=4, label=r"Input rado osservato ($\Delta t=0.04$, 20 pt)")
axes[0].plot(t_future, Y_sr_te[sample_idx, :, feat_idx], "-", color="seagreen", lw=2.2, label=r"Target futuro denso ($\Delta t=0.02$, 40 pt)")
axes[0].plot(t_future, pred_te_sr_best[sample_idx, :, feat_idx], "--", color="crimson", lw=2.0, label=f"Predizione {best_model_label} Super-Res (x2)")
axes[0].axvline(0.0, color="black", ls=":", lw=1.0)
axes[0].set_xlabel(r"Tempo fisico relativo $t - t_0$ [s]")
axes[0].set_ylabel(r"Magnetizzazione $m_1 = \langle \sigma_z^1 \rangle$")
axes[0].set_title(r"Sfida 2 – Ricostruzione Futura Densa da Input Sottocampionato ($\times 2$)")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Pannello B: Errore lungo l'orizzonte futuro (Super-Risoluzione vs Risoluzione Nativa)
mse_sr_hor = per_horizon_mse(Y_sr_va, pred_va_sr_best)
mse_nat_hor = per_horizon_mse(Y_sr_va, pred_va_nat_best)
axes[1].plot(t_future, mse_sr_hor, "-", color="crimson", lw=2, label=f"Super-Risoluzione x2 (MSE medio={mse_sr_hor.mean():.6f})")
axes[1].plot(t_future, mse_nat_hor, "--", color="navy", lw=2, label=f"Risoluzione Nativa (MSE medio={mse_nat_hor.mean():.6f})")
axes[1].set_xlabel("Orizzonte futuro [s]")
axes[1].set_ylabel("Validation MSE per timestep")
axes[1].set_title(f"Sfida 2 ({best_model_label}) – Degradazione dell'Errore lungo l'Orizzonte")
axes[1].legend(fontsize=8.5)
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
print("Grafico esportato:", save_figure(fig, "challenge2_super_resolution", subdir="04_transformer"))
plt.show()

## 10. Risoluzione della Sfida 3: Interpretabilità Fisica tramite Mappe di Attenzione (Lab 7)

Seguendo esattamente la procedura del **Lab 7 (`Transformer.ipynb`, cella `Translator`)**, estraiamo dal miglior modello **M4 (`TransformerSeq2Seq`)** i pesi di attenzione memorizzati in `last_attn_scores` eseguendo un passaggio forward in modalità eager fuori da `@tf.function` sulle finestre del Test Set:

1. **Mappe Multi-Testa di `CrossAttention` $(T_{out} \times T_{in})$**: mostrano a quali istanti del passato $T_{in}$ ciascuna testa di attenzione si aggancia per predire l'orizzonte futuro $T_{out}$, rivelando la specializzazione delle teste sulle diverse fasi delle oscillazioni quantistiche (*quantum many-body scars*).
2. **Verifica Formale della Maschera Causale Triangolare Inferiore (`CausalSelfAttention`, $T_{out} \times T_{out}$)**: certifica che tutti i pesi sopra la diagonale principale ($j > i$) sono identicamente nulli ($0.0$), impedendo qualsiasi *data leakage* dai passi futuri.
3. **Profilo Temporale Medio di Attenzione**: mette in relazione il peso medio assegnato a ciascun ritardo passato con l'andamento fisico della magnetizzazione $\langle \sigma_z^1(t) \rangle$.

In [ ]:
# =============================================================================
# 10. Sfida 3: Estrazione Eager delle Mappe di Attenzione e Verifica Causal Mask
# =============================================================================
X_te_short, Y_te_short = windows_data["short"]["test"]
X_te_sub = X_te_short[:32]
Y_te_sub = Y_te_short[:32]

attn_dict = m4_eval.get_attention_scores(X_te_sub)
cross_attn = attn_dict["cross_attention"]          # (batch, num_heads, t_out, t_in)
causal_attn = attn_dict["causal_self_attention"]   # (batch, num_heads, t_out, t_out)
enc_self_attn = attn_dict["encoder_self_attention"]  # (batch, num_heads, t_in, t_in)

# Verifica rigorosa della maschera causale triangolare inferiore (j > i deve essere 0)
upper_tri_max = float(np.max(np.abs(np.triu(causal_attn, k=1))))
row_sum_err = float(np.max(np.abs(cross_attn.sum(axis=-1) - 1.0)))
assert upper_tri_max < 1e-6, f"Violazione causal mask rilevata: {upper_tri_max}"

print("Estrazione Mappe di Attenzione completata con successo (Lab 7 Eager Pass):")
print(f"  • Cross-Attention (Decoder -> Encoder) : shape = {cross_attn.shape}")
print(f"  • Causal Self-Attention (Decoder)      : shape = {causal_attn.shape} | max |triu(k=1)| = {upper_tri_max:.2e} ✔")
print(f"  • Encoder Global Self-Attention        : shape = {enc_self_attn.shape} | max |sum(rows)-1| = {row_sum_err:.2e} ✔")

In [ ]:
# =============================================================================
# Grafico Sfida 3A: Mappe di Calore Multi-Head della Cross-Attention (Tout x Tin)
# =============================================================================
n_heads = cross_attn.shape[1]
mean_cross_heads = cross_attn.mean(axis=0)  # (num_heads, t_out, t_in)

fig, axes = plt.subplots(1, n_heads, figsize=(3.8 * n_heads, 4.4), sharey=True)
if n_heads == 1:
    axes = [axes]

for h in range(n_heads):
    im = axes[h].imshow(
        mean_cross_heads[h],
        aspect="auto",
        origin="lower",
        cmap="viridis",
        extent=[-CFG["windows"]["short"] * DT, 0.0, DT, CFG["t_out"] * DT],
    )
    axes[h].set_title(f"Head {h + 1}")
    axes[h].set_xlabel(r"Passato osservato $t - t_0$ [s]")
    if h == 0:
        axes[h].set_ylabel(r"Orizzonte futuro $\tau$ [s]")
    fig.colorbar(im, ax=axes[h], fraction=0.046, pad=0.04)

fig.suptitle(r"Sfida 3 – Mappe di Cross-Attention Multi-Testa (Decoder $\to$ Encoder, Media su Test Set)", y=1.02, fontweight="bold")
fig.tight_layout()
print("Grafico esportato:", save_figure(fig, "challenge3_cross_attention_heads", subdir="04_transformer"))
plt.show()

In [ ]:
# =============================================================================
# Grafico Sfida 3B: Verifica Maschera Causale Triangolare Inferiore vs Encoder
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.5))

im0 = axes[0].imshow(causal_attn.mean(axis=(0, 1)), aspect="auto", origin="upper", cmap="magma")
axes[0].set_title(f"Decoder Causal Self-Attention (40x40)\nTriangolo Superiore (j > i) max = {upper_tri_max:.1e}")
axes[0].set_xlabel("Key Timestep Futuro j (1..40)")
axes[0].set_ylabel("Query Timestep Futuro i (1..40)")
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(enc_self_attn.mean(axis=(0, 1)), aspect="auto", origin="upper", cmap="viridis")
axes[1].set_title("Encoder Global Self-Attention (10x10)\nContesto Passato Bidirezionale")
axes[1].set_xlabel("Key Timestep Passato k (1..10)")
axes[1].set_ylabel("Query Timestep Passato q (1..10)")
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

fig.tight_layout()
print("Grafico esportato:", save_figure(fig, "challenge3_causal_mask_verification", subdir="04_transformer"))
plt.show()

In [ ]:
# =============================================================================
# Grafico Sfida 3C: Profilo Temporale Medio di Attenzione e Oscillazione Fisica
# =============================================================================
t_in_short = CFG["windows"]["short"]
past_lags_s = np.arange(-t_in_short + 1, 1) * DT
future_s = np.arange(1, CFG["t_out"] + 1) * DT

head_profiles = mean_cross_heads.mean(axis=1)  # (num_heads, t_in)
global_profile = head_profiles.mean(axis=0)    # (t_in,)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.3))

# Pannello A: peso medio assegnato a ciascuno dei 10 timestep passati da ogni testa
for h in range(n_heads):
    axes[0].plot(past_lags_s, head_profiles[h], "o--", alpha=0.75, label=f"Head {h + 1}")
axes[0].plot(past_lags_s, global_profile, "k-s", lw=2.4, label="Media Multi-Head")
axes[0].axhline(1.0 / t_in_short, color="gray", ls=":", label=r"Uniforme ($1/T_{in}$)")
axes[0].set_xlabel(r"Istante passato osservato $t - t_0$ [s]")
axes[0].set_ylabel(r"Peso medio di Cross-Attention $\bar{\alpha}_k$")
axes[0].set_title("Specializzazione delle Teste sugli Istanti del Passato")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Pannello B: sovrapposizione tra traiettoria fisica <sigma_z^1> e intensita' di attenzione
w_idx = 0
y_hat_w = m4_eval.predict_free_running(X_te_sub[w_idx : w_idx + 1])[0]
axes[1].plot(past_lags_s, X_te_sub[w_idx, :, 0], "o-", color="navy", lw=2, label=r"Passato osservato $m_1$")
axes[1].plot(future_s, Y_te_sub[w_idx, :, 0], "-", color="seagreen", lw=2, label=r"Target futuro $m_1$")
axes[1].plot(future_s, y_hat_w[:, 0], "--", color="crimson", lw=2, label="Predizione M4 Free-Running")
axes[1].axvline(0.0, color="black", ls=":", lw=1.0)
axes[1].set_xlabel(r"Tempo fisico relativo $t - t_0$ [s]")
axes[1].set_ylabel(r"Magnetizzazione $m_1 = \langle \sigma_z^1 \rangle$")
axes[1].set_title("Allineamento Fisico tra Pesi di Attenzione e Oscillazione Quantistica")
axes[1].legend(loc="upper left", fontsize=8)
axes[1].grid(True, alpha=0.3)

ax_bar = axes[1].twinx()
ax_bar.bar(past_lags_s, cross_attn[w_idx].mean(axis=(0, 1)), width=0.012, color="darkorange", alpha=0.32, label="Peso Cross-Attention")
ax_bar.set_ylabel("Peso Cross-Attention", color="darkorange")

fig.tight_layout()
print("Grafico esportato:", save_figure(fig, "challenge3_attention_mean_profile", subdir="04_transformer"))
plt.show()

## 11. Analisi Disaggregata Dettagliata (Variabili, Qubit, Coppie e Coerenza Quantistica)

Valutiamo la distribuzione dell'errore predittivo del miglior modello Transformer sulla finestra `short`:
1. **Magnetizzazioni (10) vs Correlazioni (45)** lungo i 40 passi futuri.
2. **MSE per Singolo Qubit ($q_1, \dots, q_{10}$)** lungo la catena atomica di Rydberg.
3. **Matrice di Errore delle Correlazioni $10 \times 10$** per ciascuna coppia $(i, j)$.
4. **Controllo di Positività Quantistica (`check_quantum_probabilities`)**: verifica che le probabilità congiunte a due corpi $P(s_i, s_j) = \frac{1}{4}(1 + s_i m_i + s_j m_j + s_i s_j c_{ij})$ predette dal modello rispettino la meccanica quantistica ($P \ge 0$).

In [ ]:
# =============================================================================
# 11. Analisi Disaggregata ed Esportazione Grafici
# =============================================================================
best_short_row = results_df[results_df["window"] == "short"].sort_values("val_mse").iloc[0]
best_short_id = f"{best_short_row['model_name']}_{best_short_row['hp_name']}_short"
best_short_model = trained_models[best_short_id]

X_va_short, Y_va_short = windows_data["short"]["val"]
Y_pred_best = predict(best_short_model, X_va_short)

# 1. Errori per qubit
qubit_errs = per_qubit_mse(Y_va_short, Y_pred_best)

# 2. Matrice correlazioni 10x10
corr_mat = corr_pair_mse_matrix(Y_va_short, Y_pred_best)

# 3. Errore temporale per gruppi (Magnetizzazioni vs Correlazioni)
group_errs = per_horizon_group_mse(Y_va_short, Y_pred_best)

# 4. Verifica fisica delle probabilita' quantistiche a due corpi
q_prob_report = check_quantum_probabilities(Y_pred_best.reshape(-1, N_FEATURES))
print(f"Modello analizzato: {best_short_id}")
print(f"Controllo positivita' quantistica P(s_i, s_j) >= 0 sulle predizioni:")
print(f"  • Minima probabilita' congiunta : {q_prob_report['min_probability']:.4f}")
print(f"  • Tasso di violazioni fisiche   : {q_prob_report['violation_rate']:.6%}")

fig = plt.figure(figsize=(14, 4.2))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1.2, 1.2])

# Pannello A: Errore per qubit
ax0 = fig.add_subplot(gs[0])
ax0.bar(range(1, N_QUBITS + 1), qubit_errs, color="steelblue", edgecolor="navy", alpha=0.85)
ax0.set_xlabel("Indice Qubit")
ax0.set_ylabel("MSE")
ax0.set_title("MSE Magnetizzazioni per Qubit")
ax0.set_xticks(range(1, N_QUBITS + 1))
ax0.grid(True, axis="y", alpha=0.3)

# Pannello B: Heatmap correlazioni 10x10
ax1 = fig.add_subplot(gs[1])
im = ax1.imshow(corr_mat, cmap="viridis", interpolation="nearest")
ax1.set_title(r"Matrice MSE Correlazioni $c_{ij}$")
ax1.set_xlabel("Qubit $j$")
ax1.set_ylabel("Qubit $i$")
ax1.set_xticks(range(N_QUBITS))
ax1.set_xticklabels(range(1, N_QUBITS + 1), fontsize=8)
ax1.set_yticks(range(N_QUBITS))
ax1.set_yticklabels(range(1, N_QUBITS + 1), fontsize=8)
fig.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)

# Pannello C: Evoluzione temporale Mag vs Corr
ax2 = fig.add_subplot(gs[2])
tt = np.arange(1, CFG["t_out"] + 1) * DT
ax2.plot(tt, group_errs["mag"], label="Magnetizzazioni (10)", color="navy", lw=2)
ax2.plot(tt, group_errs["corr"], label="Correlazioni (45)", color="darkorange", lw=2)
ax2.set_title("MSE Orizzonte: Mag vs Corr")
ax2.set_xlabel("Orizzonte futuro [s]")
ax2.set_ylabel("MSE")
ax2.legend()
ax2.grid(True, alpha=0.3)

fig.tight_layout()
print("Grafico esportato:", save_figure(fig, "disaggregated_error_analysis", subdir="04_transformer"))
plt.show()

## 12. Validazione Statistica su 3 Seed ($t$ di Student) e Salvataggio Checkpoint

Come in `03_rnn.ipynb`, validiamo la configurazione Transformer vincente su **3 repliche con seed indipendenti (`[42, 43, 44]`)** per calcolare l'intervallo di confidenza al 95% secondo la distribuzione **$t$ di Student** ($t_{0.975, 2} \approx 4.303$). Per efficienza computazionale, riutilizziamo direttamente la replica `seed = 42` già addestrata nella Sezione 5 ed eseguiamo solo le due repliche aggiuntive (`seed = 43` e `seed = 44`).

Salviamo infine il checkpoint `.keras` in `checkpoints/best_transformer_model.keras` e verifichiamo la procedura di ripristino per il notebook finale `05_comparison_and_inference.ipynb`.

In [ ]:
# =============================================================================
# 12. Repliche Statistiche su 3 Seed (42, 43, 44) e Salvataggio Checkpoint
# =============================================================================
replica_seeds = CFG["replica_seeds"]  # [42, 43, 44]
seed_mses = []
seed_maes = []

best_short_type = best_short_row["model_type"]
best_short_label = best_short_row["model_name"]
best_short_hp = best_short_row["hp_name"]

best_replica_model = best_short_model
best_replica_val = float(best_short_row["val_mse"])

print(f"Validazione statistica su seed {replica_seeds} per {best_short_label} ({best_short_hp}, finestra 'short')...")

for s in replica_seeds:
    if s == 42:
        # Riutilizza la replica seed=42 gia' addestrata nella Sezione 5
        mse_42 = float(best_short_row["val_mse"])
        mae_42 = float(best_short_row["val_mae"])
        seed_mses.append(mse_42)
        seed_maes.append(mae_42)
        print(f"  Seed {s} (riutilizzato da Sez. 5) -> Val MSE: {mse_42:.6f} | Val MAE: {mae_42:.6f}")
        continue

    set_seeds(s)
    rep_cfg = make_config(model=best_short_type, t_in=10, seed=s, **hp_configs[best_short_hp])
    rep_model = build_transformer_from_config(rep_cfg, t_in=10)

    _ = train_hybrid(
        rep_model,
        windows_data["short"]["train"][0], windows_data["short"]["train"][1],
        windows_data["short"]["val"][0], windows_data["short"]["val"][1],
        rep_cfg,
        verbose=0
    )

    val_ev = evaluate(windows_data["short"]["val"][1], predict(rep_model, windows_data["short"]["val"][0]))
    seed_mses.append(val_ev["mse"])
    seed_maes.append(val_ev["mae"])
    print(f"  Seed {s} -> Val MSE: {val_ev['mse']:.6f} | Val MAE: {val_ev['mae']:.6f}")

    if val_ev["mse"] < best_replica_val:
        best_replica_val = val_ev["mse"]
        best_replica_model = rep_model

stats_mse = summarize(seed_mses)
stats_mae = summarize(seed_maes)

print("\n" + "=" * 65)
print(f"RISULTATO FINALE MODELLI TRANSFORMER (3 Repliche Seed {replica_seeds}):")
print(f"  Validation MSE : {format_mean_ci(seed_mses, digits=6)} (95% CI Student-t)")
print(f"  Validation MAE : {format_mean_ci(seed_maes, digits=6)} (95% CI Student-t)")
print("=" * 65)

# Salvataggio del checkpoint del modello migliore
ckpt_path = PATHS["checkpoints"] / "best_transformer_model.keras"
best_replica_model.save(ckpt_path)
print(f"\nModello Transformer migliore salvato con successo in: {ckpt_path}")

# Sanity check: ricaricamento del modello salvato con CUSTOM_OBJECTS
loaded_model = keras.models.load_model(ckpt_path, custom_objects=CUSTOM_OBJECTS)
re_val = evaluate(windows_data["short"]["val"][1], predict(loaded_model, windows_data["short"]["val"][0]))
assert np.isclose(re_val["mse"], best_replica_val, atol=1e-5), "Discrepanza nel modello ricaricato!"
print("Verifica ricaricamento checkpoint per inferenza finale: SUPERATA ✔")

## 13. Sincronizzazione dei Risultati e Push su GitHub (Multi-Utente)

Questa cella finale rileva automaticamente l'autore attivo in base alla chiave segreta Colab (`GIT_E_TOKEN` per Ernesto -> branch `dev-ernesto`, `GIT_R_TOKEN` per Riccardo -> branch `dev-riccardo`), traccia i log numerici in `results/`, le figure esportate in `figures/`, il notebook e il codice, ed esegue il push sul branch corretto senza conflitti.

In [ ]:
# =============================================================================
# Sincronizzazione dei Risultati e Push su GitHub (Multi-Utente)
# =============================================================================
import os
from pathlib import Path

target_repo = Path("/content/drive/MyDrive/DL-Quantum") if Path("/content/drive/MyDrive/DL-Quantum").exists() else REPO_PATH

print(f"--- SINCRONIZZAZIONE SUL BRANCH: {CURRENT_BRANCH} (Autore: {AUTHOR_NAME}) ---")

# 1. Configura l'identità Git dell'autore attivo
!git -C "{target_repo}" config user.name "{AUTHOR_NAME}"
!git -C "{target_repo}" config user.email "{AUTHOR_EMAIL}"

# 2. Allineamento preventivo con eventuali nuovi commit remoti (evita non-fast-forward)
!git -C "{target_repo}" fetch origin
!git -C "{target_repo}" pull --rebase --autostash origin {CURRENT_BRANCH}

# 3. Aggiunge i risultati numerici, figure, notebook e codice al tracking Git
!git -C "{target_repo}" add results/ figures/ notebooks/ src/
commit_msg = f"feat: aggiornamento risultati da Colab [{CURRENT_BRANCH}] di {AUTHOR_NAME}"
!git -C "{target_repo}" commit -m "{commit_msg}"
!git -C "{target_repo}" status -s

# 4. Push su GitHub con il token personale rilevato
if GIT_TOKEN:
    print(f"\nPush automatico in corso sul branch '{CURRENT_BRANCH}' con token di {AUTHOR_NAME}...")
    push_out = !git -C "{target_repo}" push https://{GIT_TOKEN}@github.com/Riccardo-27/DL-Quantum.git {CURRENT_BRANCH}
    print("\n".join(push_out))

    # Verifica esito del comando di push
    push_err = any("rejected" in l.lower() or "error:" in l.lower() or "fatal:" in l.lower() for l in push_out)
    if push_err:
        print(f"\n[ATTENZIONE] Il push è stato respinto. Risolvi eventuali divergenze con '!git -C \"{target_repo}\" pull --rebase --autostash origin {CURRENT_BRANCH}' e riprova.")
    else:
        print(f"\n✔ Sincronizzazione completata con successo su '{CURRENT_BRANCH}'!")
else:
    if IN_COLAB:
        print("\n[ATTENZIONE] Nessun token rilevato per il push automatico su Colab.")
        print("  - Se sei Ernesto: crea il segreto 'GIT_E_TOKEN' nell'icona 🔑 a sinistra.")
        print("  - Se sei Riccardo: crea il segreto 'GIT_R_TOKEN' nell'icona 🔑 a sinistra.")
    else:
        print(f"\nAmbiente locale: per il push esegui 'git push origin {CURRENT_BRANCH}' da terminale.")